# Road Accidents France — Machine Learning

This notebook builds classification models to predict injury severity from accident features.

**Approach:** Temporal validation — trained on 2023 data, tested on 2024 data.
This simulates a real production setting where a model trained on past data predicts future events.

**Target variable (`grav`):**
- 1 — Uninjured
- 2 — Killed
- 3 — Hospitalized
- 4 — Minor injury

**Models compared:**
1. Dummy Classifier (baseline)
2. Logistic Regression
3. Random Forest
4. Neural Network
5. Binary classifier — Killed vs. rest (to investigate class separability)

**Data:** output of `cleaning.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

df_train = pd.read_csv('accidents_france_2023_clean_for_ML.csv')
df_test = pd.read_csv('accidents_france_2024_clean_for_ML.csv')

print(f"Train set (2023): {df_train.shape}")
print(f"Test set  (2024): {df_test.shape}")

## 1. Class Distribution

The dataset is heavily imbalanced — fatalities represent only ~2.7% of cases.
This is a key challenge for the modeling.

In [ ]:
print("Class distribution (train set 2023):")
dist = df_train['grav'].value_counts(normalize=True).round(3)
dist.index = ['Uninjured (1)', 'Minor injury (4)', 'Hospitalized (3)', 'Killed (2)']
print(dist)

## 2. Preprocessing

- Drop identifier columns (no predictive value, potential data leakage)
- Encode categorical columns with LabelEncoder
- Fit encoder on both train and test to handle unseen values

In [ ]:
# Separate target
y_train = df_train['grav']
y_test = df_test['grav']

# Drop target and identifier columns (no predictive value + data leakage risk)
cols_to_drop = ['grav', 'id_usager', 'id_vehicule', 'Num_Acc',
                'num_veh_x', 'num_veh_y', 'com', 'datetime']
X_train = df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns])
X_test = df_test.drop(columns=[c for c in cols_to_drop if c in df_test.columns])

# Encode categorical columns
# Fit on both sets to handle values present in test but not in train
for col in X_train.select_dtypes(include='object').columns:
    le = LabelEncoder()
    le.fit(pd.concat([X_train[col], X_test[col]]).astype(str))
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

print(f"Number of features: {X_train.shape[1]}")
print(f"Features: {X_train.columns.tolist()}")

## 3. Baseline — Dummy Classifier

Always predict the most frequent class. Any useful model must clearly outperform this.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

print("=== Dummy Classifier (baseline) ===")
print(classification_report(y_test, dummy.predict(X_test), zero_division=0))

## 4. Logistic Regression

Simple linear baseline model. `class_weight='balanced'` compensates for class imbalance.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))

## 5. Random Forest

Ensemble method — 100 trees trained on random subsets of data and features.
`class_weight='balanced'` gives more weight to rare classes (fatalities).

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))

## 6. Confusion Matrix — Random Forest

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)
labels = ['Uninjured (1)', 'Killed (2)', 'Hospitalized (3)', 'Minor injury (4)']

px.imshow(cm, x=labels, y=labels,
          labels=dict(x='Predicted', y='Actual'),
          title='Confusion Matrix — Random Forest (tested on 2024)',
          height=500)

## 7. Neural Network

In [ ]:
X_train_tensor = torch.FloatTensor(X_train.values)
X_test_tensor = torch.FloatTensor(X_test.values)
y_train_tensor = torch.LongTensor(y_train.values - 1)  # classes must start at 0
y_test_tensor = torch.LongTensor(y_test.values - 1)

# 1. Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.FloatTensor(X_train_scaled)
X_test_tensor = torch.FloatTensor(X_test_scaled)

# 2. Add class weights to the loss function
class_counts = y_train.value_counts().sort_index()
weights = 1.0 / class_counts
weights = weights / weights.sum()
class_weights = torch.FloatTensor(weights.values)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# DataLoader handles batching automatically
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

In [ ]:
class AccidentNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(AccidentNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

model = AccidentNet(input_size=X_train.shape[1], num_classes=4)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        y_hat = model(X_batch)          # forward pass
        loss = criterion(y_hat, y_batch) # compute loss
        optimizer.zero_grad()            # reset gradients
        loss.backward()                  # backpropagation
        optimizer.step()                 # update weights
        total_loss += loss.item()
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor)
    y_pred_classes = torch.argmax(y_pred, dim=1).numpy() + 1  # back to 1-4

from sklearn.metrics import classification_report
print(classification_report(y_test.values, y_pred_classes))

## 8. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False).head(20).reset_index()
importances.columns = ['feature', 'importance']

px.bar(importances, x='importance', y='feature', orientation='h',
       title='Top 20 Feature Importances (Gini impurity reduction)',
       height=600)

**Observations:**
- No single feature dominates — the highest importance (Age) reaches only ~9.4%
- Geographic features (`lat`, `long`, `dep`) rank high, likely capturing local road danger levels
- Engineered features (`hour`, `is_night`, `is_weekend`) contribute, validating their creation
- `secu1` (safety equipment) and `catv` (vehicle type) are among the most informative behavioral features

## 9. Binary Classifier — Killed vs. Rest

**Motivation:** The recall on fatalities is consistently low in the multiclass setting.
We test whether a dedicated binary classifier can better separate fatalities from other outcomes.
If recall remains low even here, it definitively proves that the available features
cannot discriminate fatal accidents — regardless of the model.

In [ ]:
y_train_binary = (y_train == 2).astype(int)  # 1 = Killed, 0 = everything else
y_test_binary = (y_test == 2).astype(int)

rf_binary = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_binary.fit(X_train, y_train_binary)
y_pred_binary = rf_binary.predict(X_test)

print("=== Binary Classifier — Killed (1) vs. Rest (0) ===")
print(classification_report(y_test_binary, y_pred_binary,
                             target_names=['Not killed (0)', 'Killed (1)']))

## Binary Classifier Results — Killed vs. Rest

**Results:** Despite focusing exclusively on the killed vs. not-killed problem 
with `class_weight='balanced'`, the recall on fatalities reaches only **0.01** — 
meaning the model correctly identifies only 1% of fatal accidents.

**What this proves:**
This is the key finding of this project. Even in a simplified binary setting 
specifically designed to maximize detection of fatalities, the model fails almost 
completely. This is not a modeling failure — it is a **data failure**.

The features available in the public BAAC dataset simply do not contain enough 
signal to distinguish fatal accidents from non-fatal ones. The most discriminating 
factors for road fatalities — blood alcohol level, actual speed at impact, driver 
fatigue, reaction time — are absent from the data.

**Precision vs. Recall tradeoff:**
The model achieves 0.52 precision on fatalities — meaning that when it does 
predict "killed", it is right about half the time. But with a recall of 0.01, 
it almost never makes that prediction, defaulting instead to "not killed" 
for virtually every case.

**Conclusion:** No amount of feature engineering or model tuning can compensate 
for fundamentally missing data. This project demonstrates that public road 
accident data, while rich in circumstantial variables, lacks the behavioral 
and physiological information needed to reliably predict fatality risk.

## 9. Results Summary & Conclusion

## Results Analysis

### Overview

The table below summarizes the performance of all models tested, evaluated on 
the 2024 test set (temporal validation):

| Model               | Accuracy | Macro F1 |
|---------------------|----------|----------|
| Dummy Classifier    | 0.41     | 0.15     |
| Logistic Regression | 0.39     | 0.31     |
| Neural Network      | 0.64     | 0.45     |
| Random Forest       | 0.65     | 0.47     |

Accuracy alone is misleading here due to class imbalance — a model predicting 
"Uninjured" for every case would reach 41% accuracy without learning anything. 
We therefore focus on **Macro F1**, which treats all classes equally regardless 
of their frequency.

---

### Model Comparison

**Dummy Classifier** confirms our baseline: always predicting the majority class 
yields 41% accuracy and a macro F1 of 0.15. Any useful model must clearly 
outperform this.

**Logistic Regression** achieves a lower accuracy than the Dummy (0.39) but a 
significantly higher macro F1 (0.31). This apparent paradox is explained by 
`class_weight='balanced'`: the model sacrifices overall accuracy to better detect 
rare classes, particularly fatalities (recall of 0.50 on class 2). However, its 
overall predictive power remains limited — linear models cannot capture the 
complex interactions between accident features.

**Neural Network and Random Forest** perform nearly identically — accuracy of 
0.64 and 0.65, macro F1 of 0.45 and 0.47 respectively. This is consistent with 
the machine learning literature, which consistently shows that tree-based models 
and neural networks perform comparably on tabular data, both outperforming linear 
models on non-linear problems.

---

### The Fatality Prediction Problem

Across all four models, recall on fatalities (class 2) remains critically low:

| Model               | Precision (Killed) | Recall (Killed) |
|---------------------|--------------------|-----------------|
| Logistic Regression | 0.06               | 0.50            |
| Neural Network      | 0.67               | 0.00            |
| Random Forest       | 0.42               | 0.03            |

In a road safety context, **recall on fatalities is the most critical metric** — 
failing to identify a real fatal accident is far more costly than a false alarm. 
None of the models achieve acceptable recall on this class.

This failure is not a modeling failure. Three fundamentally different model 
families — linear, tree-based, and neural network — all reach the same conclusion. 
The binary classifier (Killed vs. Rest) further confirms this: even when the 
entire modeling effort is focused on separating fatalities from non-fatalities, 
recall remains near zero.

The root cause is a **data limitation**: the most discriminating factors for 
road fatalities — blood alcohol level, actual speed at impact, driver fatigue, 
reaction time — are absent from the public BAAC dataset. No amount of model 
tuning can compensate for fundamentally missing information.

---

### Temporal Validation

Training on 2023 and evaluating on 2024 simulates a real production setting 
where a model must generalize to future, unseen data. The consistency of results 
across both years suggests the models capture stable structural patterns in 
French road accident data, rather than overfitting to year-specific noise.